In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install kiwipiepy
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from kiwipiepy import Kiwi
import re

# Kiwi 객체 초기화
kiwi = Kiwi()

# 전처리 함수 정의
def preprocess_text(text):
    if pd.isna(text):
        return ""
    spacing_corrected_text = kiwi.space(text)
    clean_text = re.sub(r'[^가-힣0-9\s.,?!]', '', spacing_corrected_text)
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    return clean_text

# 1. 원본 데이터 로드
# AI Hub와 카카오브레인 JIT 데이터를 활용하여 학습량을 보강할 수 있습니다[cite: 29, 30].
data = pd.read_csv('./data/filtered_huggingface_dataset.csv')

# 2. 원본 데이터 전처리
print("원시 데이터 전처리 중...")
tqdm.pandas()
data['standard_form'] = data['standard_form'].progress_apply(preprocess_text)
data['dialect_form'] = data['dialect_form'].progress_apply(preprocess_text)
print("전처리 완료.")

# 3. 데이터셋 분리 (핵심 단계)
# 데이터 누수를 막기 위해 양방향 변환 전에 원본 데이터를 train, validation, test로 나눕니다.
train_df, temp_df = train_test_split(data, test_size=0.2, random_state=2003)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=2003)

print(f"Train 데이터셋 크기: {len(train_df)}")
print(f"Validation 데이터셋 크기: {len(val_df)}")
print(f"Test 데이터셋 크기: {len(test_df)}")

# 4. 각 데이터셋에 대해 양방향 데이터 생성
def create_bidirectional_data(df):
    jeju_to_std = pd.DataFrame({
        'input_text': '[제주] ' + df['dialect_form'],
        'target_text': df['standard_form']
    })
    std_to_jeju = pd.DataFrame({
        'input_text': '[표준] ' + df['standard_form'],
        'target_text': df['dialect_form']
    })
    # 각 방향의 데이터를 합치고 무작위로 섞습니다.
    # 각 세트 내에서 섞기 때문에 데이터 누수 문제 없음.
    return pd.concat([jeju_to_std, std_to_jeju]).sample(frac=1).reset_index(drop=True)

print("양방향 데이터 생성 및 결합 중...")
train_combined = create_bidirectional_data(train_df)
val_combined = create_bidirectional_data(val_df)
test_combined = create_bidirectional_data(test_df)
print("양방향 데이터 생성 및 결합 완료.")

# 5. 결과 확인
print("\n최종 Train 데이터셋 예시:")
print(train_combined.head())
print("\n최종 Test 데이터셋 예시:")
print(test_combined.head())

# 6. CSV 파일로 저장
train_combined.to_csv('./data/preprocessed_train_data.csv', index=False, encoding='utf-8-sig')
val_combined.to_csv('./data/preprocessed_val_data.csv', index=False, encoding='utf-8-sig')
test_combined.to_csv('./data/preprocessed_test_data.csv', index=False, encoding='utf-8-sig')

원시 데이터 전처리 중...


100%|██████████| 796935/796935 [13:13<00:00, 1003.81it/s]


전처리 완료.
Train 데이터셋 크기: 637548
Validation 데이터셋 크기: 79693
Test 데이터셋 크기: 79694
양방향 데이터 생성 및 결합 중...
양방향 데이터 생성 및 결합 완료.

최종 Train 데이터셋 예시:
                                        input_text  \
0                     [제주] 무슨 바위 바위 있는 그대로 허영 그 위의   
1                                 [제주] 아 그래서 다시 굳언   
2  [제주] 대박이지? 순차적으로 한 겡이 그거. 그 겡이 원래 내가 처음 봤을 때는 이   
3                               [제주] 동백꽃도 먹어 나 수가?   
4                       [제주] 너 오면 들어보켄 이것만 잠굴 수 있나   

                                target_text  
0                    무슨 바위 바위 있는 그대로해서 그 위에  
1                              아 그래서 다시 굳었어  
2  대박이지? 순차적으로 한 게 그거야. 그게 원래 내가 처음 봤을 때는 이  
3                                동백꽃도 먹었어요?  
4                     너 오면 들어본대 이것만 잠굴 수 있나  

최종 Test 데이터셋 예시:
                 input_text          target_text
0           [제주] 보난 막 해 싸졌댄         보니깐 막 흐트러졌다고
1   [제주] 공연허고 밤마다 예 최고급으로 행  공연하고 밤마다 예 최고급으로 해서
2             [표준] 시집 잘 갔지.            시집 잘 갔주게.
3  [표준] 며칠 새 막 저 추워져 가지고 이젠    매칠 새 막 저 얼어 전이 이젠
